In [5]:
import alphamap
from alphamap.importing import import_spectronaut_data
from alphamap.sequenceplot import plot_peptide_traces, get_plot_data

# Load your Spectronaut AlphaMap export
data = import_spectronaut_data(
    "C:/Users/conno/Desktop/Final Year School/Final Year Project/data_raw/ptm_closed_search_alphamap_export.tsv"
)

# Check it loaded correctly
print(data.shape)
print(data.columns.tolist())

(162629, 3)
['all_protein_ids', 'modified_sequence', 'naked_sequence']


In [8]:
# Check what protein ID format looks like
data['all_protein_ids'].head(10)

0    P62937;A0A075B767;Q9Y536;A0A075B759;A0A0B4J2A2...
1    P62937;A0A075B767;Q9Y536;A0A075B759;A0A0B4J2A2...
2    P62937;A0A075B767;P0DN37;Q9Y536;A0A075B759;A0A...
3                                           A0A075B767
4    A0A075B767;P0DN37;Q9Y536;A0A075B759;A0A0B4J2A2...
5                      P0DPK3;P0DPK4;Q7Z3S9;A0A096LNW5
6                                           A0A096LNW5
7                                           A0A096LNW5
8                                    A0A0B4J2D5;P0DPI2
9                                    A0A0B4J2D5;P0DPI2
Name: all_protein_ids, dtype: object

In [9]:
# Filter to your three proteins of interest
proteins_of_interest = ["P01584", "Q9Y6K9", "P21796"]

data_filtered = data[
    data['all_protein_ids'].str.contains('|'.join(proteins_of_interest), na=False)
]

print(data_filtered.shape)
print(data_filtered['all_protein_ids'].value_counts())


(118, 3)
all_protein_ids
P01584    43
P21796    41
Q9Y6K9    34
Name: count, dtype: int64


In [11]:
import inspect
print(inspect.signature(get_plot_data))

(protein, df, fasta)


In [13]:
import requests

def download_fasta(uniprot_id, save_path):
    url = f"https://www.uniprot.org/uniprot/{uniprot_id}.fasta"
    response = requests.get(url)
    with open(save_path, 'w') as f:
        f.write(response.text)
    print(f"Downloaded {uniprot_id}")

# Download all three
download_fasta("P01584", "C:/Users/conno/Desktop/Final Year School/Final Year Project/pdb/P01584.fasta")
download_fasta("Q9Y6K9", "C:/Users/conno/Desktop/Final Year School/Final Year Project/pdb/Q9Y6K9.fasta")
download_fasta("P21796", "C:/Users/conno/Desktop/Final Year School/Final Year Project/pdb/P21796.fasta")

Downloaded P01584
Downloaded Q9Y6K9
Downloaded P21796


In [19]:
from alphamap.sequenceplot import get_plot_data, plot_peptide_traces, import_fasta

il1b_fasta = import_fasta(
    "C:/Users/conno/Desktop/Final Year School/Final Year Project/pdb/P01584.fasta"
)

il1b_data = get_plot_data(
    protein = "P01584",
    df = data_filtered,
    fasta = il1b_fasta
)

print(il1b_data)

ValueError: Organism C:/Users/conno/Desktop/Final Year School/Final Year Project/pdb/P01584.fasta is not available. Please select one of the following: ['Human', 'Mouse', 'Rat', 'Cow', 'Zebrafish', 'Drosophila', 'Caenorhabditis elegans', 'Slime mold', 'Arabidopsis thaliana', 'Rice', 'Escherichia coli', 'Bacillus subtilis', 'Saccharomyces cerevisiae', 'SARS-CoV', 'SARS-CoV2']

In [18]:
import alphamap.importing as ai
print(dir(ai))

['StringIO', 'Union', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'convert_ap_mq_mod', 'convert_diann_mq_mod', 'convert_fragpipe_mq_mod', 'extract_rawfile_unique_values', 'import_alphapept_data', 'import_data', 'import_diann_data', 'import_fragpipe_data', 'import_maxquant_data', 'import_spectronaut_data', 'os', 'pd', 're', 'read_file']


In [20]:
il1b_fasta = import_fasta("Human")

il1b_data = get_plot_data(
    protein = "P01584",
    df = data_filtered,
    fasta = il1b_fasta
)

print(il1b_data)

.. done


AttributeError: 'DataFrame' object has no attribute 'unique_protein_id'

In [21]:
print(data_filtered.columns.tolist())
print(data.columns.tolist())

['all_protein_ids', 'modified_sequence', 'naked_sequence']
['all_protein_ids', 'modified_sequence', 'naked_sequence']


In [22]:
from alphamap.importing import import_data

# Use import_data instead
data2 = import_data(
    "C:/Users/conno/Desktop/Final Year School/Final Year Project/data_raw/ptm_closed_search_alphamap_export.tsv",
    software = "spectronaut"
)

print(data2.columns.tolist())
print(data2.shape)

TypeError: import_data() got an unexpected keyword argument 'software'

In [23]:
import inspect
print(inspect.signature(import_data))

(file: str, sample: Union[str, list, NoneType] = None, verbose: bool = True, dashboard: bool = False) -> pandas.core.frame.DataFrame


In [24]:
data2 = import_data(
    file = "C:/Users/conno/Desktop/Final Year School/Final Year Project/data_raw/ptm_closed_search_alphamap_export.tsv"
)

print(data2.columns.tolist())
print(data2.shape)

Import Spectronaut output
['all_protein_ids', 'modified_sequence', 'naked_sequence']
(162629, 3)


In [25]:
# Rename and filter in one go
data_ready = data.rename(columns={
    'all_protein_ids': 'unique_protein_id',
    'modified_sequence': 'modified_sequence',
    'naked_sequence': 'naked_sequence'
})

# Filter to your three proteins
data_ready = data_ready[
    data_ready['unique_protein_id'].str.contains("P01584|Q9Y6K9|P21796", na=False)
]

# Now try plotting directly
il1b_data = get_plot_data(
    protein = "P01584",
    df = data_ready,
    fasta = il1b_fasta
)

print(il1b_data)

KeyError: 'start'

In [27]:
from pyteomics import fasta
import numpy as np

# Load human fasta to get protein sequences
protein_sequences = {}
for entry in fasta.read("C:/Users/conno/Desktop/Final Year School/Final Year Project/pdb/P01584.fasta"):
    protein_sequences["P01584"] = entry.sequence
for entry in fasta.read("C:/Users/conno/Desktop/Final Year School/Final Year Project/pdb/Q9Y6K9.fasta"):
    protein_sequences["Q9Y6K9"] = entry.sequence
for entry in fasta.read("C:/Users/conno/Desktop/Final Year School/Final Year Project/pdb/P21796.fasta"):
    protein_sequences["P21796"] = entry.sequence

# Calculate start/end positions by matching peptide to protein sequence
def add_positions(df, protein_id, protein_seq):
    df_prot = df[df['unique_protein_id'] == protein_id].copy()
    starts, ends = [], []
    for pep in df_prot['naked_sequence']:
        pos = protein_seq.find(pep)
        starts.append(pos)
        ends.append(pos + len(pep) - 1)
    df_prot['start'] = starts
    df_prot['end'] = ends
    return df_prot

il1b_ready  = add_positions(data_ready, "P01584", protein_sequences["P01584"])
ikbkg_ready = add_positions(data_ready, "Q9Y6K9", protein_sequences["Q9Y6K9"])
vdac1_ready = add_positions(data_ready, "P21796", protein_sequences["P21796"])

print(il1b_ready[['unique_protein_id','naked_sequence','start','end']].head())

      unique_protein_id                        naked_sequence  start  end
20836            P01584                         QAASVVVAMDKLR     62   74
20837            P01584                         QAASVVVAMDKLR     62   74
20838            P01584                    CSFQDLDLCPLDGGIQLR     33   50
20839            P01584  ALHLQGQDMEQQVVFSMSFVQGEESNDKIPVALGLK    143  178
20840            P01584  ALHLQGQDMEQQVVFSMSFVQGEESNDKIPVALGLK    143  178


In [28]:
il1b_data = get_plot_data(
    protein = "P01584",
    df = il1b_ready,
    fasta = il1b_fasta
)

fig = plot_peptide_traces(
    df = il1b_data,
    uniprot_id = "P01584",
    highlight_positions = [171]
)

fig.show()

KeyError: 'all_protein_ids'

In [29]:
il1b_ready['all_protein_ids'] = il1b_ready['unique_protein_id']
ikbkg_ready['all_protein_ids'] = ikbkg_ready['unique_protein_id']
vdac1_ready['all_protein_ids'] = vdac1_ready['unique_protein_id']

# Try again
il1b_data = get_plot_data(
    protein = "P01584",
    df = il1b_ready,
    fasta = il1b_fasta
)

fig = plot_peptide_traces(
    df = il1b_data,
    uniprot_id = "P01584",
    highlight_positions = [171]
)

fig.show()

AttributeError: 'DataFrame' object has no attribute 'PTMsites'